# ④ 생성 — 질문에 답하기
담당: 생성 · 코드: `rag/generation/generation.py`

03 검증의 `artifacts/eval_set.jsonl` 질문마다 02 검색으로 청크를 찾고 `gpt-5-mini` 가 답합니다.
답변 · 검색 근거 · 토큰 · 지연 시간을 `artifacts/answers.jsonl` 로 저장합니다 → 05 평가가 읽습니다.
`.env` 에 Langfuse 키가 있으면 질문마다 Langfuse Cloud 에 추적 `answer-question`(검색 `retrieve-context` + 생성 `generate-answer` — 모델 · 토큰 · 비용)이 남고,
그 추적 id 가 `answers.jsonl` 의 `trace_id` 에 저장되어 05 평가가 점수를 붙입니다 (키가 없으면 기록하지 않고 똑같이 동작).

In [ ]:
import pandas as pd

from rag.config import ARTIFACTS, load_config, read_jsonl, write_jsonl
from rag.generation.generation import answer_questions
from rag.retrieval.retrieval import HybridIndex

cfg = load_config()

In [ ]:
eval_set = read_jsonl(ARTIFACTS / "eval_set.jsonl")
index = HybridIndex(ARTIFACTS, cfg)
answers = answer_questions(eval_set, index, cfg["llm"])
write_jsonl(answers, ARTIFACTS / "answers.jsonl")
pd.DataFrame(answers)[["id", "question", "answer", "sources", "latency_s"]]